# Rerun: FP8 only (Qwen2.5-7B on vLLM)
Runtime → Change runtime type → **L4 GPU**, then Run all.

The first run lost FP8's concurrency-64 point and its accuracy, because the client stopped on a request whose generated tokens were all special tokens (see `results/L4_quant.md`). The client is fixed.

- Installs vLLM 0.32.0 and downloads the BF16 model, about 15 GB (about 12 minutes).
- Runs a smoke test and then FP8 in full (about 25 minutes).
- About 1.5 compute units in total.
- At the end it downloads `quant_fp8.zip`.

The notebook **stops** if vLLM fails to install or the smoke test fails. In the full run, a config that fails to start is skipped and listed at the end. If anything fails, send the output of that cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('/content/lowbit-decode-gpu'):
    !git clone -q https://github.com/FredWen-S/lowbit-decode-gpu.git /content/lowbit-decode-gpu
%cd /content/lowbit-decode-gpu
!git pull -q && git log -1 --format='%h %s'

vLLM in its own virtualenv, pinned to the version of the serving runs.

In [ ]:
!pip -q install uv
!uv venv --clear --python 3.12 /content/venv-vllm
!uv pip install --python /content/venv-vllm/bin/python vllm==0.32.0 ninja 2>&1 | tail -25

In [ ]:
import os, subprocess
r = subprocess.run(['/content/venv-vllm/bin/python', '-c', 'import vllm; print(vllm.__version__)'], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])
assert r.returncode == 0, 'vllm did not install; see the install cell above'
assert os.path.exists('/content/venv-vllm/bin/ninja'), 'ninja missing in venv-vllm'

In [ ]:
from huggingface_hub import snapshot_download
for repo in ['Qwen/Qwen2.5-7B-Instruct']:
    path = snapshot_download(repo, local_dir='/content/models/' + repo.split('/')[1])
    print(path)
!pip -q install aiohttp datasets
!du -sh /content/models/*

Smoke test: FP8 with few requests, 100 MMLU items and 10 WikiText chunks.

In [ ]:
!rm -f /content/SMOKE_FAILED
!python bench/quant_sweep.py --models-dir /content/models --python /content/venv-vllm/bin/python --quick --configs fp8 --out results/quant_fp8_quick || touch /content/SMOKE_FAILED
import os
assert not os.path.exists('/content/SMOKE_FAILED'), 'smoke test failed; see the output above'

Full FP8 run.

In [ ]:
!python bench/quant_sweep.py --models-dir /content/models --python /content/venv-vllm/bin/python --configs fp8 --out results/quant_fp8

In [ ]:
!zip -qr /content/quant_fp8.zip results/quant_fp8 results/quant_fp8_quick
from google.colab import files
files.download('/content/quant_fp8.zip')